# 05 — Test du pipeline modèle (bonne pratique)

Ce notebook sert à **valider rapidement** que tout le pipeline fonctionne avant un vrai entraînement :

- import du package `src/`
- chargement du dataset
- application des transformations
- split train / validation
- création des `DataLoader`
- test d’un batch
- test du modèle `SmallCNN`
- test du modèle `ResNet18`

L’idée est simple : **si ce notebook fonctionne, le pipeline de base est sain**.


## 1) Rendre `src/` importable

Si le notebook est exécuté depuis `notebooks/`, on remonte à la racine du projet puis on ajoute `src/` à `sys.path`.


In [13]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SRC exists =", SRC_DIR.exists())


PROJECT_ROOT = /home/alouiyaz/projects/PINKCC/PINKCC_challenge_Cité_neutral_Minds
SRC exists = True


## 2) Imports du projet

On importe ici uniquement le code déjà rangé dans `src/`.


In [14]:
from collections import Counter

import torch
from torch.utils.data import Subset

from pinkcc_ct_seg.data.dataset import BrainMRIDataset
from pinkcc_ct_seg.data.transforms import get_train_transforms, get_eval_transforms
from pinkcc_ct_seg.data.split import make_train_val_split
from pinkcc_ct_seg.data.loaders import make_loaders

from pinkcc_ct_seg.models.cnn_small import SmallCNN
from pinkcc_ct_seg.models.resnet18 import build_resnet18

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: False


## 3) Définition des chemins et transformations

On travaille ici sur le dossier `Training` pour tester le pipeline.


In [15]:
DATA_DIR = PROJECT_ROOT / "data/raw/brain_mri/Training"

train_tfms = get_train_transforms(img_size=224)
eval_tfms = get_eval_transforms(img_size=224)

print("DATA_DIR =", DATA_DIR)
print("Exists =", DATA_DIR.exists())


DATA_DIR = /home/alouiyaz/projects/PINKCC/PINKCC_challenge_Cité_neutral_Minds/data/raw/brain_mri/Training
Exists = True


## 4) Dataset brut

On vérifie :
- le nombre total d’images
- la forme d’une image
- le type du label
- la distribution des classes


In [16]:
dataset = BrainMRIDataset(DATA_DIR, transform=train_tfms)

print("Dataset size:", len(dataset))

x0, y0 = dataset[0]
print("Image shape:", x0.shape)
print("Label:", y0)
print("Label dtype:", y0.dtype)

labels = [dataset[i][1].item() for i in range(len(dataset))]
print("Distribution:", Counter(labels))


Dataset size: 2870
Image shape: torch.Size([3, 224, 224])
Label: tensor(1)
Label dtype: torch.int64
Distribution: Counter({1: 2475, 0: 395})


## 5) Split train / validation

On utilise le split stratifié défini dans le projet.


In [17]:
train_idx, val_idx = make_train_val_split(labels, val_size=0.2, random_state=42)

print("Train size:", len(train_idx))
print("Val size:", len(val_idx))


Train size: 2296
Val size: 574


## 6) Création des subsets

On utilise :
- `train_tfms` pour le train
- `eval_tfms` pour la validation


In [18]:
train_dataset = BrainMRIDataset(DATA_DIR, transform=train_tfms)
eval_dataset = BrainMRIDataset(DATA_DIR, transform=eval_tfms)

train_set = Subset(train_dataset, train_idx)
val_set = Subset(eval_dataset, val_idx)

print("Train subset:", len(train_set))
print("Val subset:", len(val_set))


Train subset: 2296
Val subset: 574


## 7) DataLoaders

Pour un simple test de pipeline, `weighted=False` suffit.


In [19]:
pin_memory = torch.cuda.is_available()

train_loader, val_loader = make_loaders(
    train_set,
    val_set,
    batch_size=32,
    num_workers=0,
    weighted=False,
    pin_memory=pin_memory,
)

xb, yb = next(iter(train_loader))

print("Batch x shape:", xb.shape)
print("Batch y shape:", yb.shape)
print("Labels min/max:", yb.min().item(), yb.max().item())
print("Labels dtype:", yb.dtype)
print("Batch distribution:", Counter(yb.tolist()))


Batch x shape: torch.Size([32, 3, 224, 224])
Batch y shape: torch.Size([32])
Labels min/max: 0 1
Labels dtype: torch.int64
Batch distribution: Counter({1: 30, 0: 2})


## 8) Test du modèle `SmallCNN`

On vérifie uniquement que le passage avant (*forward pass*) fonctionne et renvoie la bonne forme.


In [20]:
cnn_model = SmallCNN(img_size=224, num_classes=2)

cnn_out = cnn_model(xb)

print("SmallCNN output shape:", cnn_out.shape)


SmallCNN output shape: torch.Size([32, 2])


## 9) Test du modèle `ResNet18`

Même principe : on teste le `forward` pour vérifier que tout est compatible.


In [21]:
resnet_model = build_resnet18(num_classes=2, pretrained=False)

resnet_out = resnet_model(xb)

print("ResNet18 output shape:", resnet_out.shape)


ResNet18 output shape: torch.Size([32, 2])


## 10) Test sur le device (CPU/GPU)

Si un GPU est disponible, on vérifie aussi que les modèles tournent dessus.


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

xb_device = xb.to(device)

cnn_model = cnn_model.to(device)
resnet_model = resnet_model.to(device)

with torch.no_grad():
    out_cnn = cnn_model(xb_device)
    out_resnet = resnet_model(xb_device)

print("Device:", device)
print("CNN device output:", out_cnn.shape, out_cnn.device)
print("ResNet device output:", out_resnet.shape, out_resnet.device)


Device: cpu
CNN device output: torch.Size([32, 2]) cpu
ResNet device output: torch.Size([32, 2]) cpu


In [23]:
# test d'import des bibleo evaluation et training

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import torch
from torch import nn

from pinkcc_ct_seg.data.dataset import BrainMRIDataset
from pinkcc_ct_seg.data.transforms import get_train_transforms, get_eval_transforms
from pinkcc_ct_seg.data.split import make_train_val_split
from pinkcc_ct_seg.data.loaders import make_loaders

from pinkcc_ct_seg.models.resnet18 import build_resnet18
from pinkcc_ct_seg.training.engine import train_one_epoch, validate_one_epoch
from pinkcc_ct_seg.training.utils import set_seed, get_device, save_checkpoint
from pinkcc_ct_seg.evaluation.metrics import compute_metrics

## 11) Résultat attendu

Si tout est correct, tu dois avoir :

- dataset chargé
- images sous forme `torch.Size([3, 224, 224])`
- labels `torch.int64`
- batchs `torch.Size([32, 3, 224, 224])`
- sorties modèles `torch.Size([32, 2])`

À partir de là, tu peux passer à l’entraînement réel en confiance.
